# Tasks 1–3: Web Scraping, EDA and Data Visualization

**Dataset:** Books to Scrape — public demo website

This notebook demonstrates the complete workflow from collecting public web data to exploratory analysis and visualization.

## Task 1 — Web Scraping
The scraper uses `requests` to retrieve HTML and `BeautifulSoup` to locate product cards. It extracts title, price, availability and rating. The website is designed for scraping practice.

In [ ]:
"""
TASK 1 - Web Scraping
Source: https://books.toscrape.com/
Purpose: scrape book title, price, availability and rating from public catalogue pages.

The website is a demo site specifically intended for web-scraping practice.
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

def scrape_books(max_pages=50):
    rows = []
    for page in range(1, max_pages + 1):
        url = BASE_URL.format(page)
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        cards = soup.select("article.product_pod")
        if not cards:
            break

        for card in cards:
            title_tag = card.select_one("h3 a")
            price_tag = card.select_one(".price_color")
            availability_tag = card.select_one(".availability")
            rating_tag = card.select_one("p.star-rating")

            title = title_tag.get("title", "").strip()
            price_text = price_tag.get_text(strip=True).replace("£", "")
            availability = availability_tag.get_text(" ", strip=True) if availability_tag else ""
            rating = rating_tag.get("class", ["", "Unknown"])[1] if rating_tag else "Unknown"

            rows.append({
                "title": title,
                "price_gbp": float(price_text),
                "availability": availability,
                "rating": rating,
                "page": page
            })

    return pd.DataFrame(rows)

if __name__ == "__main__":
    data = scrape_books(max_pages=50)
    data.to_csv("../data/books_scraped_full.csv", index=False)
    print(f"Saved {len(data)} records.")


## Task 2 — Exploratory Data Analysis
Questions: What is the typical book price? Which books are most expensive? How are prices distributed? Are there missing values or unusual prices?

In [ ]:
"""
TASK 2 - Exploratory Data Analysis
"""

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/books_scraped_sample.csv")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDescriptive statistics:")
print(df.describe(include="all"))

print("\nMost expensive books:")
print(df.nlargest(10, "price_gbp")[["title", "price_gbp"]])

print("\nPrice-band counts:")
print(df["price_band"].value_counts().sort_index())

# Basic hypothesis:
# H0: the mean sample price is £30 or less.
# H1: the mean sample price is greater than £30.
# The sample mean is compared with £30 descriptively.
mean_price = df["price_gbp"].mean()
print(f"\nSample mean price: £{mean_price:.2f}")
print("Descriptive conclusion:", "above £30" if mean_price > 30 else "£30 or below")

# IQR anomaly check
q1 = df["price_gbp"].quantile(0.25)
q3 = df["price_gbp"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = df[(df["price_gbp"] < lower) | (df["price_gbp"] > upper)]
print("\nIQR bounds:", lower, upper)
print("Potential outliers:")
print(outliers[["title", "price_gbp"]])


## Task 3 — Data Visualization
The visualizations focus on price distribution, highest-priced books, price bands and listing-order patterns.

In [ ]:
"""
TASK 3 - Data Visualization
"""

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/books_scraped_sample.csv")

# 1. Price distribution
plt.figure(figsize=(10,6))
plt.hist(df["price_gbp"], bins=6)
plt.title("Distribution of Book Prices")
plt.xlabel("Price (£)")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.show()

# 2. Highest prices
top10 = df.nlargest(10, "price_gbp").sort_values("price_gbp")
plt.figure(figsize=(10,6))
plt.barh(top10["title"], top10["price_gbp"])
plt.title("10 Highest-Priced Books")
plt.xlabel("Price (£)")
plt.tight_layout()
plt.show()

# 3. Price bands
df["price_band"] = pd.cut(
    df["price_gbp"],
    bins=[0,20,40,60],
    labels=["£0–£20","£20–£40","£40–£60"],
    include_lowest=True
)
counts = df["price_band"].value_counts().sort_index()
plt.figure(figsize=(8,5))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Books by Price Band")
plt.xlabel("Price Band")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.show()

# 4. Listing order
plt.figure(figsize=(10,5))
plt.plot(range(1, len(df)+1), df["price_gbp"], marker="o")
plt.title("Book Price by Scraped Listing Order")
plt.xlabel("Listing Order")
plt.ylabel("Price (£)")
plt.tight_layout()
plt.show()
